# EEGNet: Minimal Train & Test (Colab-ready)
This notebook loads labeled EEG segments, prepares splits, trains EEGNet, and evaluates it.

In [ ]:
# Colab deps (safe no-op locally)
import sys
if 'google.colab' in sys.modules:
    %pip install -q tensorflow scikit-learn

In [ ]:
# Imports
import os, sys, numpy as np
from pathlib import Path
from typing import List, Sequence, Tuple, Optional, Union
from sklearn.model_selection import train_test_split
from tensorflow.keras import utils as np_utils
import tensorflow as tf

In [ ]:
# Paths & basic config (edit BASE_DIR if running on Colab)
BASE_DIR = os.environ.get('EEG_EYE_BASE_DIR', '.')
LABELED_EEG_DATA_DIR = Path(BASE_DIR) / 'data' / 'data' / 'labeled_eeg_data'
PRE_WINDOW_MS = 500
POST_WINDOW_MS = 500

# Auto-discover available subjects (folders starting with 'P')
SUBJECTS = sorted([d.name for d in Path(LABELED_EEG_DATA_DIR).iterdir() if d.is_dir() and d.name.startswith('P')])
SUBJECTS = [s for s in SUBJECTS if (LABELED_EEG_DATA_DIR / s / f'{PRE_WINDOW_MS}_{POST_WINDOW_MS}_training_data.npz').exists()]
print(f'Found {len(SUBJECTS)} subjects:', SUBJECTS[:10], '...')

In [ ]:
# Minimal data loader with simple channel selection (names only)
from typing import Sequence, Optional, List


def load_subject_labeled_eeg(
    subject: str,
    channels: Optional[List[str]] = None,
    pre_ms: int = 500,
    post_ms: int = 500,
):
    path = LABELED_EEG_DATA_DIR / subject / f"{pre_ms}_{post_ms}_training_data.npz"
    data = np.load(str(path))

    X = data["X"]  # (trials, channels, samples)
    y = data["y"]
    channel_names = data["channel_names"].tolist()
    time = data["time"] if "time" in data.files else None

    # Slice by requested channel names (if provided)
    if channels is not None:
        missing = [ch for ch in channels if ch not in channel_names]
        if missing:
            raise ValueError(f"The following channels are not in the data: {missing}")
        indices = [channel_names.index(ch) for ch in channels]
        X = X[:, indices, :]
        channel_names = [channel_names[i] for i in indices]

    return X, y, channel_names, time


def load_labeled_eeg_data(
    subjects: Sequence[str],
    pre_ms: int = 500,
    post_ms: int = 500,
    channels: Optional[List[str]] = None,
):
    """
    Load labeled EEG segments across subjects, optionally selecting specific channel names.
    Returns concatenated X, y, the selected channel_names, and a shared time axis (if present).
    """
    X_blocks, y_blocks = [], []
    selected_channel_names = None
    time_axis = None

    for subj in subjects:
        X, y, ch_names, t = load_subject_labeled_eeg(
            subj, channels=channels, pre_ms=pre_ms, post_ms=post_ms
        )
        # Keep the first subject's channel order as canonical
        if selected_channel_names is None:
            selected_channel_names = ch_names
        X_blocks.append(X)
        y_blocks.append(y)
        if time_axis is None:
            time_axis = t

    X_out = np.concatenate(X_blocks, axis=0)
    y_out = np.concatenate(y_blocks, axis=0)
    return X_out, y_out, selected_channel_names, time_axis


def apply_memory_opts(X: np.ndarray, time_axis: Optional[np.ndarray]):
    # Downsample along time axis (samples dimension)
    if isinstance(DOWNSAMPLE, int) and DOWNSAMPLE and DOWNSAMPLE > 1:
        X = X[:, :, ::DOWNSAMPLE]
        if time_axis is not None:
            time_axis = time_axis[::DOWNSAMPLE]
    # Cast dtype to reduce memory
    if MEMORY_DTYPE == 'float16':
        X = X.astype(np.float16, copy=False)
    elif MEMORY_DTYPE == 'float32':
        X = X.astype(np.float32, copy=False)
    return X, time_axis


def prepare_data_for_eegnet(
    X: np.ndarray,
    y: np.ndarray,
    test_size: float = 0.3,
    val_size: float = 0.2,
    random_state: int = 42,
):
    # Add kernel dimension for EEGNet (channels_last: (trials, Chans, Samples, 1))
    Xp = X.reshape(X.shape[0], X.shape[1], X.shape[2], 1)
    y_flat = np.squeeze(y).astype(int)

    # First split: train+val vs test
    X_temp, X_test, y_temp, y_test = train_test_split(
        Xp,
        y_flat,
        test_size=test_size,
        shuffle=True,
        stratify=y_flat,
        random_state=random_state,
    )

    # Second split: train vs val
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp,
        y_temp,
        test_size=val_size,
        shuffle=True,
        stratify=y_temp,
        random_state=random_state,
    )

    # One-hot labels
    n_classes = int(len(np.unique(y_flat)))
    y_train_oh = np_utils.to_categorical(y_train, num_classes=n_classes)
    y_val_oh = np_utils.to_categorical(y_val, num_classes=n_classes)
    y_test_oh = np_utils.to_categorical(y_test, num_classes=n_classes)

    return X_train, X_val, X_test, y_train_oh, y_val_oh, y_test_oh

In [ ]:
import numpy as np
import tensorflow as tf
from typing import Sequence, List
from sklearn.model_selection import train_test_split

# --------------------------
# Shared utilities
# --------------------------
def _load_channels(X, ch_names, channels: List[str]):
    """Select only the requested channels and keep order."""
    idx = [ch_names.index(c) for c in channels]
    return X[:, idx, :]

# --------------------------
# 1. Inter-subject streaming
# --------------------------
def make_inter_subject_dataset(subjects: Sequence[str],
                               channels: List[str],
                               pre_window_ms: int,
                               post_window_ms: int,
                               batch_size: int = 32,
                               val_frac: float = 0.2,
                               test_frac: float = 0.2,
                               shuffle_buffer: int = 4096,
                               seed: int = 42):
    """Stream EEG trials per subject; split subjects into train/val/test."""
    
    # Build file paths
    file_list = [str(LABELED_EEG_DATA_DIR / s / f"{pre_window_ms}_{post_window_ms}_training_data.npz")
                 for s in subjects]

    # Split subjects into train/val/test
    train_files, test_files = train_test_split(file_list, test_size=test_frac, random_state=seed)
    train_files, val_files = train_test_split(train_files, test_size=val_frac/(1-test_frac), random_state=seed)

    def _generator(files):
        for fp in files:
            with np.load(fp, mmap_mode='r') as d:
                X = d['X'].astype(np.float32)
                y = np.squeeze(d['y']).astype(int)
                ch_names = d['channel_names'].tolist()
                X = _load_channels(X, ch_names, channels)
                X = np.expand_dims(X, -1)  # EEGNet expects (Chans, Samples, 1)
                for Xi, yi in zip(X, y):
                    yield Xi, tf.one_hot(yi, 2, dtype=tf.float32)

    def _make_dataset(files, shuffle=True):
        output_signature = (
            tf.TensorSpec(shape=(len(channels), None, 1), dtype=tf.float32),
            tf.TensorSpec(shape=(2,), dtype=tf.float32)
        )
        ds = tf.data.Dataset.from_generator(lambda: _generator(files), output_signature=output_signature)
        if shuffle:
            ds = ds.shuffle(shuffle_buffer, reshuffle_each_iteration=True)
        return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    return _make_dataset(train_files), _make_dataset(val_files, shuffle=False), _make_dataset(test_files, shuffle=False)

# --------------------------
# 2. Intra-subject streaming
# --------------------------
def make_intra_subject_dataset(subjects: Sequence[str],
                               channels: List[str],
                               pre_window_ms: int,
                               post_window_ms: int,
                               batch_size: int = 32,
                               val_frac: float = 0.2,
                               test_frac: float = 0.2,
                               shuffle_buffer: int = 4096,
                               seed: int = 42):
    """Split trials within each subject into train/val/test and stream."""

    file_list = [str(LABELED_EEG_DATA_DIR / s / f"{pre_window_ms}_{post_window_ms}_training_data.npz")
                 for s in subjects]

    # Precompute trial splits per file
    splits = {}
    for i, fp in enumerate(file_list):
        with np.load(fp, mmap_mode='r') as d:
            n_trials = d['y'].shape[0]
        rng = np.random.RandomState(seed + i)
        perm = rng.permutation(n_trials)
        n_test = int(round(n_trials * test_frac))
        n_val = int(round(n_trials * val_frac))
        splits[fp] = {
            'test': perm[:n_test],
            'val': perm[n_test:n_test + n_val],
            'train': perm[n_test + n_val:]
        }

    def _trial_generator(split_name):
        for fp in file_list:
            idxs = splits[fp][split_name]
            with np.load(fp, mmap_mode='r') as d:
                X = d['X'].astype(np.float32)
                y = np.squeeze(d['y']).astype(int)
                ch_names = d['channel_names'].tolist()
                X = _load_channels(X, ch_names, channels)
                X = np.expand_dims(X, -1)
                for i in idxs:
                    yield X[i], tf.one_hot(y[i], 2, dtype=tf.float32)

    def _make_dataset(split_name, shuffle=True):
        output_signature = (
            tf.TensorSpec(shape=(len(channels), None, 1), dtype=tf.float32),
            tf.TensorSpec(shape=(2,), dtype=tf.float32)
        )
        ds = tf.data.Dataset.from_generator(lambda: _trial_generator(split_name), output_signature=output_signature)
        if split_name == 'train' and shuffle:
            ds = ds.shuffle(shuffle_buffer, reshuffle_each_iteration=True)
        return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    return _make_dataset('train'), _make_dataset('val'), _make_dataset('test')

In [ ]:
# Channel selection config (set to a subset to save RAM)
# Examples:
# CHANNELS_TO_USE = ['Fz', 'Cz', 'Pz', 'Oz']
# CHANNELS_TO_USE = [0, 1, 2, 3, 4]
CHANNELS_TO_USE = None  # None keeps all channels

# Memory options
# Use 'float32' for safety; 'float16' halves memory but watch for underflow.
MEMORY_DTYPE = 'float32'  # 'float32' or 'float16'
# Downsample factor along time axis; 1 means no downsampling.
DOWNSAMPLE = 1  # try 2 or 4 on Colab to save RAM

# Training controls
BATCH = 32
USE_TF_DATA = False  # set True to train from disk via tf.data

In [ ]:
# Load and prepare data
X, y, channel_names, time_axis = load_labeled_eeg_data(
    SUBJECTS, pre_ms=PRE_WINDOW_MS, post_ms=POST_WINDOW_MS, channels=CHANNELS_TO_USE
)
X, time_axis = apply_memory_opts(X, time_axis)
print('X shape:', X.shape, '| y shape:', y.shape, '| dtype:', X.dtype)
print('Channels:', len(channel_names))
unique, counts = np.unique(np.squeeze(y), return_counts=True)
print('Label distribution:', dict(zip(unique.tolist(), counts.tolist())))

X_train, X_val, X_test, y_train, y_val, y_test = prepare_data_for_eegnet(X, y)
Chans, Samples = X.shape[1], X.shape[2]
n_classes = y_train.shape[1]
print('Train:', X_train.shape, 'Val:', X_val.shape, 'Test:', X_test.shape)
print('Classes:', n_classes, '| Chans:', Chans, '| Samples:', Samples)

In [ ]:
# EEGNet model (adapted from project models/eeg_net.py)
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Activation, Dropout, Conv2D, AveragePooling2D, SeparableConv2D, DepthwiseConv2D, BatchNormalization, SpatialDropout2D, Input, Flatten
from tensorflow.keras.constraints import max_norm

def EEGNet(nb_classes, Chans, Samples, dropoutRate=0.2, kernLength=64, F1=8, D=3, F2=16, norm_rate=0.25, dropoutType='Dropout'):
    if dropoutType == 'SpatialDropout2D':
        dropoutType = SpatialDropout2D
    elif dropoutType == 'Dropout':
        dropoutType = Dropout
    else:
        raise ValueError('dropoutType must be SpatialDropout2D or Dropout')

    input1 = Input(shape=(Chans, Samples, 1))
    block1 = Conv2D(F1, (1, kernLength), padding='same', use_bias=False)(input1)
    block1 = BatchNormalization()(block1)
    block1 = DepthwiseConv2D((Chans, 1), use_bias=False, depth_multiplier=D, depthwise_constraint=max_norm(1.))(block1)
    block1 = BatchNormalization()(block1)
    block1 = Activation('elu')(block1)
    block1 = AveragePooling2D((1, 4))(block1)
    block1 = dropoutType(dropoutRate)(block1)

    block2 = SeparableConv2D(F2, (1, 16), use_bias=False, padding='same')(block1)
    block2 = BatchNormalization()(block2)
    block2 = Activation('elu')(block2)
    block2 = AveragePooling2D((1, 8))(block2)
    block2 = dropoutType(dropoutRate)(block2)

    flatten = Flatten(name='flatten')(block2)
    dense = Dense(nb_classes, name='dense', kernel_constraint=max_norm(norm_rate))(flatten)
    softmax = Activation('softmax', name='softmax')(dense)
    return Model(inputs=input1, outputs=softmax)

In [ ]:
# Build and train
model = EEGNet(nb_classes=n_classes, Chans=Chans, Samples=Samples, dropoutRate=0.5)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

EPOCHS = 50
# Respect global BATCH override if defined earlier
BATCH = globals().get('BATCH', 64)
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH,
    verbose=1
)

In [ ]:
# Evaluate
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test accuracy: {test_acc:.4f} | Test loss: {test_loss:.4f}')

In [ ]:
# Optional: predict and show a simple confusion matrix
from sklearn.metrics import confusion_matrix
y_pred = model.predict(X_test, verbose=0)
y_true_cls = np.argmax(y_test, axis=1)
y_pred_cls = np.argmax(y_pred, axis=1)
cm = confusion_matrix(y_true_cls, y_pred_cls)
print('Confusion matrix:\n', cm)